In [26]:
import pandas as pd

In [27]:
df_query = pd.read_csv('input/diphtheria_element_stats.csv',na_filter=False)
len(df_query)

365

In [28]:
df_query

,model_name,question_package,disease_type,field_name,max_value,count_of_values,case_counts
0,DiseaseSurveillanceModel_General,CLINICAL,Diphtheria,ACF_NO,NO,1,71
1,DiseaseSurveillanceModel_General,ADMINISTRATIVE,Diphtheria,ADDITIONAL_SOURCE_CASE_REPORT_ELR,ELR,71,71
2,DiseaseSurveillanceModel_General,ADMINISTRATIVE,Diphtheria Non Respiratory Toxogenic,ADDITIONAL_SOURCE_CASE_REPORT_ELR,ELR,98,101
3,DiseaseSurveillanceModel_General,ADMINISTRATIVE,Diphtheria,ADDITIONAL_SOURCE_CASE_REPORT_PAPER_LAB,PAPER_LAB,1,71
4,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria Non Respiratory Toxogenic,ADDRESS_TYPE_HOME,Home,1,101
...,...,...,...,...,...,...,...
360,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria Non Respiratory Toxogenic,UPDATE_ADDRESS_AT_REPORTING_NO,NO,101,101
361,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria,WORK_WITHIN_LAST_6_MONTHS_NO,NO,1,71
362,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria Non Respiratory Toxogenic,WORK_WITHIN_LAST_6_MONTHS_NO,NO,1,101
363,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria,ZIP,99999,71,71


In [29]:
def join_all_except_last(question,answer):

    # Check if answer matches last part of the question
    start=len(question)-len(answer)
    if(question[start:].lower()==answer.lower()):
        return question[:start].strip('_')
    else:
        return question

In [30]:
df_query.loc[:,'parent_question_id']  = [ join_all_except_last(q,a) for q,a in zip(df_query.loc[:,'field_name'],df_query.loc[:,'max_value'])]

In [31]:
df_query

,model_name,question_package,disease_type,field_name,max_value,count_of_values,case_counts,parent_question_id
0,DiseaseSurveillanceModel_General,CLINICAL,Diphtheria,ACF_NO,NO,1,71,ACF
1,DiseaseSurveillanceModel_General,ADMINISTRATIVE,Diphtheria,ADDITIONAL_SOURCE_CASE_REPORT_ELR,ELR,71,71,ADDITIONAL_SOURCE_CASE_REPORT
2,DiseaseSurveillanceModel_General,ADMINISTRATIVE,Diphtheria Non Respiratory Toxogenic,ADDITIONAL_SOURCE_CASE_REPORT_ELR,ELR,98,101,ADDITIONAL_SOURCE_CASE_REPORT
3,DiseaseSurveillanceModel_General,ADMINISTRATIVE,Diphtheria,ADDITIONAL_SOURCE_CASE_REPORT_PAPER_LAB,PAPER_LAB,1,71,ADDITIONAL_SOURCE_CASE_REPORT
4,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria Non Respiratory Toxogenic,ADDRESS_TYPE_HOME,Home,1,101,ADDRESS_TYPE
...,...,...,...,...,...,...,...,...
360,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria Non Respiratory Toxogenic,UPDATE_ADDRESS_AT_REPORTING_NO,NO,101,101,UPDATE_ADDRESS_AT_REPORTING
361,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria,WORK_WITHIN_LAST_6_MONTHS_NO,NO,1,71,WORK_WITHIN_LAST_6_MONTHS
362,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria Non Respiratory Toxogenic,WORK_WITHIN_LAST_6_MONTHS_NO,NO,1,101,WORK_WITHIN_LAST_6_MONTHS
363,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria,ZIP,99999,71,71,ZIP


In [32]:
# Group by parent_question_id and model_name to get the num of values reported
df_group = df_query.groupby(by=['parent_question_id','model_name','question_package','disease_type','case_counts'])['count_of_values'].sum().reset_index(name='count_of_values')

In [33]:
print(df_group)

                parent_question_id                        model_name  \
0                              ACF  DiseaseSurveillanceModel_General   
1    ADDITIONAL_SOURCE_CASE_REPORT  DiseaseSurveillanceModel_General   
2    ADDITIONAL_SOURCE_CASE_REPORT  DiseaseSurveillanceModel_General   
3                     ADDRESS_TYPE  DiseaseSurveillanceModel_General   
4                     ADDRESS_TYPE  DiseaseSurveillanceModel_General   
..                             ...                               ...   
332    UPDATE_ADDRESS_AT_REPORTING  DiseaseSurveillanceModel_General   
333      WORK_WITHIN_LAST_6_MONTHS  DiseaseSurveillanceModel_General   
334      WORK_WITHIN_LAST_6_MONTHS  DiseaseSurveillanceModel_General   
335                            ZIP  DiseaseSurveillanceModel_General   
336                            ZIP  DiseaseSurveillanceModel_General   

    question_package                          disease_type  case_counts  \
0           CLINICAL                            Diphtheria  

In [34]:
df_group.loc[:,'fill_rate'] = [round(c*100.0/t,3) for c,t in zip(df_group.loc[:,'count_of_values'],df_group.loc[:,'case_counts'])]

In [35]:
df_group 

,parent_question_id,model_name,question_package,disease_type,case_counts,count_of_values,fill_rate
0,ACF,DiseaseSurveillanceModel_General,CLINICAL,Diphtheria,71,1,1.408
1,ADDITIONAL_SOURCE_CASE_REPORT,DiseaseSurveillanceModel_General,ADMINISTRATIVE,Diphtheria,71,72,101.408
2,ADDITIONAL_SOURCE_CASE_REPORT,DiseaseSurveillanceModel_General,ADMINISTRATIVE,Diphtheria Non Respiratory Toxogenic,101,98,97.030
3,ADDRESS_TYPE,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria,71,2,2.817
4,ADDRESS_TYPE,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria Non Respiratory Toxogenic,101,1,0.990
...,...,...,...,...,...,...,...
332,UPDATE_ADDRESS_AT_REPORTING,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria Non Respiratory Toxogenic,101,101,100.000
333,WORK_WITHIN_LAST_6_MONTHS,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria,71,1,1.408
334,WORK_WITHIN_LAST_6_MONTHS,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria Non Respiratory Toxogenic,101,1,0.990
335,ZIP,DiseaseSurveillanceModel_General,DEMOGRAPHIC,Diphtheria,71,71,100.000


In [36]:
df_group = df_group[['model_name','question_package','disease_type','parent_question_id','case_counts','count_of_values','fill_rate']]

In [37]:
df_group.to_csv('output/fill_rate_diphtheria_fields_20260915.csv',index=False)